# Visión: candidatos de balón en los huecos del ATD (Colab, solo inferencia)

Corre `src/ball_detect.py` con **WASB** (Tarashima et al., BMVC 2023; [github.com/nttcom/WASB-SBDT](https://github.com/nttcom/WASB-SBDT), código MIT) y sus **pesos públicos de fútbol** (entrenados por los autores con ISSIA-CNR; nota de licencia en `docs/PLAN.md` §6). **Solo inferencia**, con el preprocesado y el postprocesado de evaluación del propio WASB. Nada se entrena, ajusta ni revisa con el vídeo.

**Reglas (no negociables):**
- **Gemini desactivado** en este notebook (*Configuración → IA de Colab*). No uses el chat ni "explicar error" con datos.
- **No mires las detecciones contra el vídeo** ni dibujes heatmaps o cajas sobre fotogramas: está prohibido verificar la salida con el vídeo.
- Aquí entran solo el vídeo táctico (desde la carpeta compartida de los organizadores) y `vision_bundle.zip` (segmentos + código). La homografía y el ATD se quedan en local.

**Pasos**
1. *Entorno de ejecución → Cambiar tipo → GPU.*
2. En Drive, sobre la carpeta compartida: *Organizar → Añadir acceso directo → Mi unidad*. Pon la ruta del `.mp4` táctico en `VIDEO` (celda 1).
3. Sube `external/vision/vision_bundle.zip` a `MyDrive/vision/` (o en la celda 2).
4. Celda 3: WASB y sus pesos. Celda 4: smoke test (22 fotogramas). Celda 5: pasada completa (lee el vídeo entero en orden; reanudable).
5. Celda 6: descarga `detections.csv` y `status.json` a `external/vision/out/` en local.

In [ ]:
VIDEO = '/content/drive/MyDrive/BAR-ATH_tactical.mp4'   # acceso directo en la raíz de Mi unidad
WORK = '/content/drive/MyDrive/vision'
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(WORK, exist_ok=True)
assert os.path.exists(VIDEO), 'ajusta VIDEO'
print(os.path.getsize(VIDEO) / 1e9, 'GB')

In [ ]:
import os, shutil
if not os.path.exists(f'{WORK}/vision_bundle.zip'):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/vision_bundle.zip')
!rm -rf /content/vision && mkdir -p /content/vision && unzip -q {WORK}/vision_bundle.zip -d /content/vision && ls -R /content/vision

In [ ]:
# WASB (código MIT) y sus pesos de fútbol, el mismo archivo que descarga su setup_weights.sh
!pip -q install omegaconf gdown
!git clone -q https://github.com/nttcom/WASB-SBDT /content/WASB-SBDT && git -C /content/WASB-SBDT rev-parse HEAD
W = f'{WORK}/wasb_soccer_best.pth.tar'
if not os.path.exists(W):
    !gdown -q 1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l -O {W}
!ls -la {W}

In [ ]:
# Smoke test: 20 tripletes con step 1 (22 fotogramas) de punta a punta
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --out /content/smoke --smoke
!cat /content/smoke/status.json; wc -l /content/smoke/detections.csv

In [ ]:
# Pasada completa. Reanudable: vuelve a ejecutar la celda (y la 3) si Colab se desconecta
%cd /content/vision
!python -m src.ball_detect --video "{VIDEO}" --segments segments.csv --wasb /content/WASB-SBDT --weights {W} --out {WORK}/out 2>&1 | tee -a {WORK}/log.txt
!cat {WORK}/out/status.json

In [ ]:
from google.colab import files
for f in ('detections.csv', 'status.json'):
    files.download(f'{WORK}/out/{f}')
files.download(f'{WORK}/log.txt')